# HMM で品詞タガーを作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/10_sequence_labeling/hmm_tagger.ipynb)

NLP クックブック 10-06 ／ 系列ラベリング

隠れマルコフモデル (HMM) による古典的な品詞タグ付けを学習・評価して、ベースラインと比べたい。

In [1]:
import nltk
nltk.download("treebank", quiet=True)
nltk.download("universal_tagset", quiet=True)
from nltk.corpus import treebank
from nltk.tag import DefaultTagger, UnigramTagger, hmm, tnt

sents = list(treebank.tagged_sents(tagset="universal"))
train, test = sents[:3500], sents[3500:]
print(len(train), len(test), train[0][:6])

baseline = UnigramTagger(train, backoff=DefaultTagger("NOUN"))   # 各単語に最頻の品詞を付けるだけ
trainer = hmm.HiddenMarkovModelTrainer()
tagger = trainer.train_supervised(train, estimator=lambda fd, bins: nltk.LidstoneProbDist(fd, 0.01, bins))

tnt_tagger = tnt.TnT(unk=DefaultTagger("NOUN"), Trained=True)   # trigram HMM + 未知語は名詞とみなす
tnt_tagger.train(train)

print("ユニグラム (最頻タグ):   ", round(baseline.accuracy(test), 4))
print("HMM (bigram):            ", round(tagger.accuracy(test), 4))
print("TnT (trigram + 未知語処理):", round(tnt_tagger.accuracy(test), 4))

3500 414 [('Pierre', 'NOUN'), ('Vinken', 'NOUN'), (',', '.'), ('61', 'NUM'), ('years', 'NOUN'), ('old', 'ADJ')]
ユニグラム (最頻タグ):    0.9357
HMM (bigram):             0.9129
TnT (trigram + 未知語処理): 0.9401


In [2]:
for sent in ["They can fish .", "I saw her duck .", "Time flies like an arrow ."]:
    print(tagger.tag(sent.split()))

[('They', 'PRON'), ('can', 'VERB'), ('fish', 'X'), ('.', '.')]
[('I', 'PRON'), ('saw', 'VERB'), ('her', 'PRON'), ('duck', 'VERB'), ('.', '.')]
[('Time', 'NOUN'), ('flies', 'NOUN'), ('like', 'ADP'), ('an', 'DET'), ('arrow', 'NOUN'), ('.', '.')]


In [3]:
# 未知語と既知語に分けて正解率を見る
vocab = {w for s in train for w, _ in s}
known = unknown = known_ok = unknown_ok = 0
for s in test:
    for (w, gold), (_, pred) in zip(s, tagger.tag([w for w, _ in s])):
        if w in vocab:
            known += 1; known_ok += gold == pred
        else:
            unknown += 1; unknown_ok += gold == pred
print(f"既知語 {known_ok / known:.3f} ({known} 語) / 未知語 {unknown_ok / unknown:.3f} ({unknown} 語)")

既知語 0.969 (9376 語) / 未知語 0.363 (953 語)


- 「各単語の最頻タグ」だけで 9 割を超える。品詞タグ付けの難しさは、残りの曖昧な語と未知語にある。
- 素朴な HMM は未知語の出力確率をうまく見積もれず、未知語でほぼ当たらない。そのせいで全体ではユニグラムのベースラインに負けることすらある。
- HMM は単語の綴りを見ないのが弱点。接尾辞や大文字などの特徴を使える CRF やニューラルモデルが有利な理由。